# 09A 让实际执行结果进入学习反馈

本节把第八节的导读接到一轮完整学习：读懂一个知识点，做小练习，运行，得到提示，修改，再次验证，保存后继续。

我们从一个带有典型边界错误的起始版本出发，完整保留“要求—尝试—证据—改进”的过程。先区分三个层次：程序正常退出不等于所有用例通过；所有随课用例通过也不等于已经掌握。

**相对第八节的增量**：沿用资料身份、检索坐标、Harness 和导读证据，只新增可执行练习契约与逐项测试记录。09B 再增加提示和学习状态，09C 才做跨仓库迁移验收。

**贯穿案例**

学生实现一个排序函数：过滤非正分、稳定保留同分顺序、限制返回数量。`run_exercise` 在隔离进程实际执行学生文件，保存每条测试的输入、预期和实际结果；零分项与同分项是能区分错误条件的反例，反馈必须指向失败用例，不能只说“排序不对”。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：一个零分项，区分 >0 与 >=0。

课件页：L09.03-S01、L09.03-S02。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, tempfile, subprocess, inspect, hashlib
from datetime import datetime, timezone
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
PROJECT = ROOT / "09 综合实践二 练习反馈与跨仓库迁移/代码/学习反馈"
SOURCE_PROJECT = ROOT / "08 综合实践一 资料与仓库导读/代码/学习助手"
sys.path.insert(0,str(SOURCE_PROJECT/"src"))
sys.path.insert(0,str(PROJECT/"src"))
from exercise_lab import LearningStore,run_exercise,catalog,CATALOG,verified_feedback
from corpus import Corpus
from tutor_runtime import TutorRuntime
from IPython.display import display,Code
print("默认离线；只执行课程自编的小练习，不执行资料仓库中的任意代码。")


默认离线；只执行课程自编的小练习，不执行资料仓库中的任意代码。


## 1. 把知识点写成可以检查的要求

练习不是“实现一个智能体”这样的大任务。本例只要求过滤、稳定排序和条数限制。先说明输入前提，再给出可观察的输出。

稳定排序：两项分数相同，保持它们原来的先后顺序。limit 必须是正整数；Python 的 bool 是 int 的子类，题目明确不接受它，因此要单独处理。


In [2]:
exercise=catalog()["rank-positive"]
print(exercise["prompt"])
for test in exercise["tests"]:print(test["id"],test.get("expected",test.get("expected_exception")))
print((PROJECT/"exercises/rank-positive/student.py").read_text(encoding='utf-8'))
# 先预测：哪个用例会失败？哪些用例碰巧能通过？


实现 solve(items, limit)：items 是含 id 和数值 score 的列表，过滤 score<=0 的项，按 score 降序，保持同分原顺序，最多返回 limit 项。limit 必须是正整数，布尔值不算整数。其他输入按上述前提提供。
filter-zero [{'id': 'hit', 'score': 2}]
descending [{'id': 'b', 'score': 4}, {'id': 'a', 'score': 1}]
stable-tie [{'id': 'b', 'score': 3}, {'id': 'a', 'score': 3}]
limit-one [{'id': 'a', 'score': 2}]
empty []
invalid-limit ValueError
boolean-limit ValueError
"""练习模板：按题意完成函数；先预测下方测试会失败在哪里。"""
def solve(items, limit):
    # items 为包含 id 和 score 的字典列表；limit 为正整数。
    # TODO：过滤零分，按分数降序，保持同分原顺序，再限制条数。
    return items[:limit]



## 2. 实际运行，比一句评价多了哪些证据

执行器在另一个 Python 进程加载明确选择的练习文件，逐个调用 solve，比较值和类型，并保存源码与题目哈希。每次尝试都有独立 ID，失败也保留。

这里的隔离模式、空环境和超时只是本机教学措施，不是操作系统沙箱。仅用于你信任的课程练习或自己编写的本地提交；Agent 没有任意 run_code 工具，导入的第三方仓库仍只读。


In [3]:
display(Code(inspect.getsource(run_exercise),language="python"))
print((PROJECT/"src/exercise_worker.py").read_text(encoding='utf-8'))


def run_exercise(exercise_id,submission):
    """由 Notebook/CLI 操作者选择本地文件；Agent 工具不接收任意代码或执行路径。"""
    exercise=catalog()[exercise_id];path=Path(submission)
    if path.is_symlink():raise ValueError('练习文件不接受符号链接')
    path=path.resolve()
    allowed=[(PROJECT/'exercises'/exercise_id).resolve(),(ROOT/'.local/student-submissions').resolve()]
    if not any(path.is_relative_to(root) for root in allowed):raise PermissionError('仅运行随课练习或操作者指定的本地提交目录')
    if path.suffix!='.py' or path.stat().st_size>16000:raise ValueError('只执行 16 KB 以内的 Python 练习')
    source=path.read_bytes();spec_hash=hashlib.sha256(CATALOG.read_bytes()).hexdigest()
    with tempfile.TemporaryDirectory() as workspace:
        # 不将模型凭据传入子进程。隔离模式/超时不是操作系统沙箱。
        try:
            completed=subprocess.run([sys.executable, '-X', 'utf8','-I','-S',str(PROJECT/'src/exercise_worker.py'),str(path)],
                input=json.dumps({'tests':exercise['tests']}),text=True,capture_output=True,
                cwd=workspace,env={},timeout=4, encoding='utf-8')
            if completed.returncode!=0:result={'passed':False,'tests':[],'execution_error':'process_exit'}
            else:result=json.loads(completed.stdout)
        except subprocess.TimeoutExpired:result={'passed':False,'tests':[],'execution_error':'timeout'}
    return {'attempt_id':uuid.uuid4().hex,'exercise_id':exercise_id,'source_sha256':hashlib.sha256(source).hexdigest(),
        'catalog_sha256':spec_hash,'recorded_at':datetime.now(timezone.utc).isoformat(),
        'execution':'actual_python_subprocess','result':result}

"""独立 Python 进程执行操作者明确选择的本地练习。不是不可信代码沙箱。"""
import contextlib,copy,importlib.util,io,json,sys

def typed_equal(a,b):
    if type(a) is not type(b):return False
    if isinstance(b,dict):return a.keys()==b.keys() and all(typed_equal(a[k],v) for k,v in b.items())
    if isinstance(b,list):return len(a)==len(b) and all(typed_equal(x,y) for x,y in zip(a,b))
    return a==b

def run(path,tests):
    captured=io.StringIO();rows=[]
    with contextlib.redirect_stdout(captured),contextlib.redirect_stderr(captured):
        spec=importlib.util.spec_from_file_location('course_submission',path)
        module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
        for case in tests:
            try:
                actual=module.solve(*copy.deepcopy(case['args']))
                # 本练习约定 JSON 可表示的输出；不偷偷转字符串当作正确类型。
                json.dumps(actual,allow_nan=False)
                passed='expected' in case and typed_equal(actual,case['expected'])
                row={'id':cas

In [4]:
attempt=run_exercise("rank-positive",PROJECT/"exercises/rank-positive/student.py")
for test in attempt["result"]["tests"]:print(test)
print("整体通过:",attempt["result"]["passed"],"源码版本:",attempt["source_sha256"][:12])
assert not attempt["result"]["passed"]


{'id': 'filter-zero', 'passed': False, 'actual': [{'id': 'zero', 'score': 0}, {'id': 'hit', 'score': 2}]}
{'id': 'descending', 'passed': False, 'actual': [{'id': 'a', 'score': 1}, {'id': 'b', 'score': 4}]}
{'id': 'stable-tie', 'passed': True, 'actual': [{'id': 'b', 'score': 3}, {'id': 'a', 'score': 3}]}
{'id': 'limit-one', 'passed': True, 'actual': [{'id': 'a', 'score': 2}]}
{'id': 'empty', 'passed': True, 'actual': []}
{'id': 'invalid-limit', 'passed': False, 'actual': []}
{'id': 'boolean-limit', 'passed': False, 'actual': []}
整体通过: False 源码版本: 63f1d90ae6eb


## 3. 只改一个条件，解释结果

在下面这个小列表里，切片之前要做哪些事？先独立补全处理过程，再展开教学修订示例。代码注释要说明条件为什么存在，避免仅把代码翻译成中文。

给出的测试并不穷尽所有合法输入。例如非零负分也应被过滤；下面补一个新输入，检查是否把“过滤零分”误写成“只判断不等于零”。


In [5]:
items=[{"id":"negative","score":-1},{"id":"b","score":3},{"id":"a","score":3}]
# TODO：过滤后稳定降序，再最多取两项。
prediction=["b","a"]
print("我的预测:",prediction)


我的预测: ['b', 'a']


In [6]:
selected=sorted((x for x in items if x["score"]>0),key=lambda x:-x["score"])[:2]
assert [x["id"] for x in selected]==prediction
print(selected)
print((PROJECT/"exercises/rank-positive/teaching_revision.py").read_text(encoding='utf-8'))
passed=run_exercise("rank-positive",PROJECT/"exercises/rank-positive/teaching_revision.py")
assert passed["result"]["passed"]
print("随课用例通过:",len(passed["result"]["tests"]))


[{'id': 'b', 'score': 3}, {'id': 'a', 'score': 3}]
"""课程教学修订示例：先独立尝试，再阅读。"""
def solve(items, limit):
    if type(limit) is not int or limit < 1:
        raise ValueError('limit 必须是正整数')
    positive = [item for item in items if item['score'] > 0]
    # Python 排序稳定；同分元素的原始顺序保留。
    ordered = sorted(positive, key=lambda item: item['score'], reverse=True)
    return ordered[:limit]



随课用例通过: 7


### 一个零分项，区分 >0 与 >=0

配套课件：L09.03-S01、L09.03-S02。

已有样例的负分项能检查过滤，但不能单独区分严格大于零和大于等于零。增加恰好为零的边界，同时保留同分两项，检查稳定排序与截断顺序。直接调用课程自编的可信 teaching_revision.py。

In [7]:
import importlib.util
case_path = PROJECT / "exercises/rank-positive/teaching_revision.py"
case_spec = importlib.util.spec_from_file_location("classroom_rank_solution", case_path)
case_module = importlib.util.module_from_spec(case_spec)
case_spec.loader.exec_module(case_module)
case_items = [{"id":"zero", "score":0}, {"id":"b", "score":3}, {"id":"a", "score":3}]
case_correct = case_module.solve(case_items, 3)
case_mutant = sorted((x for x in case_items if x["score"] >= 0), key=lambda x:-x["score"])[:3]
print("正确输出:", [x["id"] for x in case_correct])
print("把 > 改成 >= 的输出:", [x["id"] for x in case_mutant])
assert [x["id"] for x in case_correct] == ["b", "a"]
assert [x["id"] for x in case_mutant] == ["b", "a", "zero"]


正确输出: ['b', 'a']
把 > 改成 >= 的输出: ['b', 'a', 'zero']


**结果解读**

正确实现排除 zero，并保持 b 在 a 前；错误比较会多出 zero。limit 设为 3 是刻意的：若只取前 2 项，错误过滤可能被截断掩盖。

**小练习**

将 limit 改为 2，再说明为什么“两份输出一样”并不证明比较符号写对。

<details><summary>完成后展开参考分析</summary>

零分项排在后面，被前两项截断，所以该设置不能区分两种实现。测试必须让目标错误在输出中可见。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 4. 把练习契约与尝试证据保存为可复核产物

最后不只保留屏幕输出，还要写出题目标识、源码版本、逐项结果和结论边界。这样重新打开 Notebook 时，不必重跑也能知道这次判断基于哪一份提交。


In [8]:
out=ROOT/".local/student-runs/l09";out.mkdir(parents=True,exist_ok=True)
audit={
    "artifact":"exercise-contract-and-attempt-audit",
    "recorded_at":datetime.now(timezone.utc).isoformat(),
    "exercise_id":"rank-positive",
    "learning_objective":"按契约过滤正分项，稳定降序并应用正整数 limit",
    "contract":{"inputs":["items: 含 id 与 score 的列表","limit: 正整数且不接受 bool"],
                "observable_result":"返回过滤、稳定排序并截断后的新列表"},
    "attempt":{"attempt_id":attempt["attempt_id"],"source_sha256":attempt["source_sha256"],
               "passed":attempt["result"]["passed"],"tests":attempt["result"]["tests"]},
    "verified_revision":{"attempt_id":passed["attempt_id"],"source_sha256":passed["source_sha256"],
                         "passed":passed["result"]["passed"],"tests":passed["result"]["tests"]},
    "conclusion_boundary":"随课用例通过只证明当前契约样例通过，不等于学习者已经掌握或覆盖全部合法输入"
}
json_path=out/"exercise-contract-and-attempt-audit.json"
md_path=out/"exercise-contract-and-attempt-audit.md"
json_path.write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding="utf-8")
failed_ids=[t["id"] for t in audit["attempt"]["tests"] if not t["passed"]]
md_path.write_text("\n".join(["# 练习契约与尝试审计","",f"- 练习：{audit['exercise_id']}",
    f"- 起始版本失败用例：{', '.join(failed_ids)}",f"- 修订版本随课用例通过：{audit['verified_revision']['passed']}",
    f"- 结论边界：{audit['conclusion_boundary']}"]),encoding="utf-8")
print(json_path);print(md_path)


.local\student-runs\l09\exercise-contract-and-attempt-audit.json
.local\student-runs\l09\exercise-contract-and-attempt-audit.md


## 回顾

一个小练习需要输入前提、行为要求、可观察结果和有区分力的用例。实测记录保留源码版本和每项结果。测试通过的范围必须说清楚，不能让模型文字替代执行证据。
